# Mühlenanwendungsfall — Modellierung

Kurz gehalten, absichtlich: Auswahl und Begründung des Modells stehen in
Abschnitt 2.2.3 und 2.2.4. Kapitel 4 **trainiert das dort gewählte statische
MLP neu**, es begründet es nicht erneut.

Trainiert wird auf den Abtastzeilen der stationären Phasen des **ersten
Feldexperiments** — dem Datensatz, der den Stellgrößenraum am breitesten
abdeckt und am Anfang des Beobachtungszeitraums steht. Bewertet werden drei
Ströme mit demselben Modell:

| Strom | Rolle |
|---|---|
| DOE1 | Trainingsbasis, Bezug für die Fehlernormierung |
| DOE2 | unabhängige Generalisierungsreferenz, ~1,5 Jahre später |
| Produktion | der Betrieb, über den die Adaption später läuft |

Die Gegenüberstellung dieser drei ist die Zahl, an der Kapitel 4 hängt: der
dokumentierte Kernbefund der Vorarbeit lautet, dass der Fehler auf der zweiten
Kampagne und der Fehler auf der Produktion **negativ korrelieren**. Hier steht
der statische Ausgangspunkt dieser Aussage; geprüft wird sie in
*mill_adaptation*.

In [ ]:
FORCE_RECOMPUTE = False
IS_FINAL = False

In [ ]:
# Kennungen der Vorstufe; None -> jeweils neuestes Artefakt aus mill_data.
RUN_ID = None       # data_stream
TRAIN_ID = None     # data_train1 / data_train2
DOE_ID = None       # data_doe1 / data_doe2

SEED = 42
SMOOTH_TRAIN = True   # Butterworth auf den Trainingszeilen, phasenweise
WIN_DAYS = 14         # Fenster des gleitenden RMSE
PLOT_FRAC = 0.25      # Anteil geplotteter Punkte
PLOT_YLIM = (-1e2, 1e2)

# Architektur, Epochen, Batch, Validierungsanteil kommen aus mill_train --
# eine Quelle der Wahrheit, geteilt mit mill_adaptation.

In [ ]:
# TensorFlow zuerst: sonst gehen unter Windows die statischen TLS-Slots aus
# (DLL-Fehler 1114), wenn es nach anderen nativen Bibliotheken geladen wird.
import tensorflow as tf   # noqa: F401
import os
import sys
from pathlib import Path

work_dir = Path.cwd()
base_dir = Path(os.environ.get("BASE_DIR") or next(
    p for p in (work_dir, *work_dir.parents) if (p / "pyproject.toml").exists()))
for _p in (str(base_dir), str(base_dir / "src"), str(work_dir)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

data_dir = base_dir / "data" / "mill"
model_dir = base_dir / "models"
plot_dir = base_dir / "plots"
results_dir = base_dir / "results"
for _d in (data_dir, model_dir, plot_dir, results_dir):
    os.makedirs(_d, exist_ok=True)

from src.utils import mill_io as mio
from src.utils import mill_train as mt
print(f"Artefakte: {data_dir}")

In [ ]:
import numpy as np
import pandas as pd
from src.utils import run_registry as rr

rr.assert_current()
ledger = rr.run_ledger(work_dir)
data_store = rr.mill_data_store(data_dir)
model_store = rr.mill_model_store(model_dir)
plot_store = rr.mill_plot_store(plot_dir, ledger=ledger)
res_store = rr.mill_results_store(results_dir, ledger=ledger)

FEATURES, TARGET = mio.FEATURES, mio.TARGET

run_id = RUN_ID or data_store.latest_id("data_stream")
train_id = TRAIN_ID or data_store.latest_id("data_train1")
doe_id = DOE_ID or data_store.latest_id("data_doe1")
if None in (run_id, train_id, doe_id):
    raise FileNotFoundError("Kein data-Artefakt -- bitte mill_data.ipynb ausfuehren.")

stream = data_store.load("data_stream", run_id=run_id)
train_rows = data_store.load("data_train1", run_id=train_id)
test_rows = data_store.load("data_train2", run_id=train_id)
doe = {w: data_store.load(f"data_doe{w}", run_id=doe_id) for w in ("1", "2")}
print(f"data_stream={run_id}  data_train={train_id}  data_doe={doe_id}")
print(f"Produktion {len(stream)} Punkte | Trainingszeilen {len(train_rows)} "
      f"({train_rows['bp'].nunique()} Phasen) | DOE2-Zeilen {len(test_rows)}")

## Training

Zwei Dinge unterscheiden das Training vom naheliegenden Vorgehen, und beide
stehen in `mill_train`: skaliert wird auf die **festen Grenzen** der
Anlagenkonfiguration statt auf die Daten, und gesplittet wird nach
**Betriebspunkt** statt nach Zeile. Benachbarte Abtastzeilen eines stationären
Punktes sind fast identisch — ein zeilenweiser Split würde die Interpolation
innerhalb eines Punktes messen, nicht die Verallgemeinerung auf einen neuen.

Geglättet wird phasenweise: über eine Sollwertänderung hinweg zu filtern machte
aus einer Stufe eine Rampe.

In [ ]:
model_cfg = rr.RunConfig({
    "data_id": run_id, "train_id": train_id, "doe_id": doe_id,
    "arch": list(mt.ARCH), "act": mt.ACT, "opt": "adam", "lr": mt.LR,
    "epochs": mt.EPOCHS, "batch": mt.BATCH, "val_split": mt.VAL_SPLIT,
    "patience": mt.PATIENCE, "seed": SEED,
    "features": FEATURES, "target": TARGET, "train_on": "doe1",
    "scaler": "fixed_range",
    "smooth": ({"kind": "butter", "order": mt.SMOOTH_ORDER, "wn": mt.BUTTER_WN}
               if SMOOTH_TRAIN else None),
})
print(f"model_cfg = {model_cfg.id}")

In [ ]:
if model_store.exists("model", model_cfg, rekey=True) and not FORCE_RECOMPUTE:
    _m = model_store.load("model", model_cfg, rekey=True)
    model, x_scaler, y_scaler = _m["model"], _m["x_scaler"], _m["y_scaler"]
    history, report = _m.get("history"), _m.get("report", {})
    print("Modell aus Cache")
else:
    tr = train_rows
    if SMOOTH_TRAIN:
        tr = pd.concat([mt.smooth(g) for _, g in tr.groupby("bp")]).sort_index()
    X, y, report = mt.prepare(tr)
    print(f"{report['n_in']} Zeilen -> {report['n_out']} nutzbar "
          f"({report['dropped']} verworfen, "
          f"{report['n_out_of_range']} ausserhalb des Nennbereichs)")
    x_scaler, y_scaler = report["x_scaler"], report["y_scaler"]

    model = mt.build_model(X.shape[1])
    print(f"{model.count_params()} Parameter, Architektur {tuple(mt.ARCH)}")
    model, history = mt.fit_model(model, X, y, groups=tr["bp"].to_numpy(),
                                  seed=SEED, verbose=0)
    print(f"{len(history['loss'])} Epochen gelaufen "
          f"(val_loss {history['val_loss'][-1]:.5f})")
    model_store.save({"model": model, "x_scaler": x_scaler, "y_scaler": y_scaler,
                      "history": history,
                      "report": {k: v for k, v in report.items()
                                 if k not in ("x_scaler", "y_scaler")}},
                     "model", model_cfg)

## Bewertung

Alle drei Ströme werden mit demselben Modell und denselben Skalierern bewertet.
Der Fehler steht in der Einheit der Zielgröße (kWh/t) und zusätzlich normiert
auf Mittelwert und Streuung des **DOE1-Fehlers** — dieselbe Konvention wie im
synthetischen Fall und im TEP: Bezug ist der drift-freie Trainingsdatensatz.

`oor` zählt Punkte außerhalb des Nennbereichs. Das ist hier keine Aussage über
die Stichprobe, sondern über den Betrieb: eine Extrapolation ist kein Drift,
sieht im Fehler aber genauso aus.

In [ ]:
# Praediktion, Fehler und Nennbereichs-Marke eines Stroms.
def score(df: pd.DataFrame) -> pd.DataFrame:
    Xr = df[FEATURES].to_numpy(float)
    y_hat = y_scaler.inverse_transform(
        model.predict(x_scaler.transform(Xr), verbose=0)).ravel()
    out = df.copy()
    out["y_pred"] = y_hat
    out["model_error"] = df[TARGET].to_numpy(float) - y_hat
    out["oor"] = x_scaler.out_of_range(Xr)
    return out


doe_scored = {w: score(d) for w, d in doe.items()}
stream_scored = score(stream)

# Normierungsbasis: der Fehler auf der Trainingskampagne.
NORM_MEAN = float(doe_scored["1"]["model_error"].mean())
NORM_STD = float(doe_scored["1"]["model_error"].std())
for _d in (*doe_scored.values(), stream_scored):
    _d["model_error_norm"] = (_d["model_error"] - NORM_MEAN) / NORM_STD
print(f"Normierung aus DOE1: {NORM_MEAN:+.3f} +- {NORM_STD:.3f} kWh/t")

In [ ]:
data_store.save(stream_scored, "data_scored_stream", model_cfg)
for _w, _d in doe_scored.items():
    data_store.save(_d, f"data_scored_doe{_w}", model_cfg)
print("gespeichert:", ["data_scored_stream"] + [f"data_scored_doe{w}" for w in doe_scored])

### Modellfehler über der Zeit

Der zentrale Plot des Kapitels. Die grüne Kurve der Kapitel-3-Abbildungen — das
bekannte Driftsignal $c[k]$ — fehlt hier, und genau das ist der Punkt: an der
realen Anlage ist die Driftursache nicht gemessen. Ihr Stellvertreter wird in
*mill_context* erst konstruiert.

In [ ]:
import matplotlib.pyplot as plt
from src.utils import thesis_style as ts

w = ts.fig_width()

fig, ax, ax2 = ts.error_timeseries(
    stream_scored.index,
    [(stream_scored["model_error_norm"].to_numpy(), "drift_afflicted",
      "Norm. Modellfehler")],
    cd=None, xlabel="Zeit", ylim=PLOT_YLIM,
    frac=PLOT_FRAC, rng=np.random.default_rng(SEED), s=7, alpha=0.55,
    month_interval=6, date_format="%m/%y",
    legend=dict(bbox_to_anchor=[0.5, 1.16], loc="upper center", ncols=2),
)
plot_store.save_figure(fig, "model_error", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

In [ ]:
# Gleitender RMSE ueber ein ZEITfenster, nicht ueber eine Punktzahl: auf einem
# unregelmaessigen Strom ist das der einzige Verlauf, der ueber Jahre
# vergleichbar bleibt.
_e = stream_scored["model_error"]
_roll = np.sqrt((_e ** 2).rolling(f"{WIN_DAYS}D", min_periods=3).mean())

fig, ax = plt.subplots(figsize=(w, w * 0.32))
ax.plot(_roll.index, _roll.values, color=ts.C["drift_afflicted"], lw=1.4,
        label=f"Gleitender RMSE ({WIN_DAYS} Tage)")
for _w_, _col in (("1", "drift_free"), ("2", "highlight")):
    _r = float(np.sqrt(np.mean(doe_scored[_w_]["model_error"] ** 2)))
    ax.axhline(_r, color=ts.C[_col], lw=1.0, ls="--",
               label=f"DOE{_w_} ({_r:.2f} kWh/t)")
ax.set_ylabel("RMSE [kWh/t]")
ax.set_xlabel("Zeit")
ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower center",
          bbox_to_anchor=(0.5, 1.01))
ax.grid(True, axis="y", alpha=0.2)
plot_store.save_figure(fig, "model_rmse_rolling", model_cfg, final=IS_FINAL,
                       savefig_kwargs={"dpi": 300, "bbox_inches": "tight"},
                       archive_kwargs={"dpi": 300, "bbox_inches": "tight"})
plt.show()

### Zahlen

In [ ]:
def _stats(d: pd.DataFrame) -> dict:
    e = d["model_error"].to_numpy(float)
    y = d[TARGET].to_numpy(float)
    return dict(n=len(e), rmse=float(np.sqrt(np.mean(e ** 2))),
                mae=float(np.mean(np.abs(e))), bias=float(np.mean(e)),
                sigma=float(np.std(e)),
                r2=float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)),
                oor_share=float(d["oor"].mean()))


stats = pd.DataFrame({"DOE1": _stats(doe_scored["1"]),
                      "DOE2": _stats(doe_scored["2"]),
                      "Produktion": _stats(stream_scored)}).T
print(stats.to_string(float_format=lambda v: f"{v:.3f}"))

# Erste gegen zweite Haelfte des Produktionsstroms: waechst der Fehler?
_mid = stream_scored.index[0] + (stream_scored.index[-1] - stream_scored.index[0]) / 2
_h1 = stream_scored.loc[:_mid, "model_error"].to_numpy(float)
_h2 = stream_scored.loc[_mid:, "model_error"].to_numpy(float)
RMSE_H1, RMSE_H2 = float(np.sqrt(np.mean(_h1 ** 2))), float(np.sqrt(np.mean(_h2 ** 2)))
print(f"\nProduktion, RMSE erste Haelfte {RMSE_H1:.2f} / zweite {RMSE_H2:.2f} kWh/t "
      f"(Faktor {RMSE_H2 / RMSE_H1:.2f})")

## Ergebnisse speichern

In [ ]:
from src.utils import results_export as rx

ledger.bind("model", parents={"data": run_id})
_keys = [(k, k, "num") for k in ("n", "rmse", "mae", "bias", "sigma", "r2",
                                 "oor_share")]
(rx.ResultDoc()
 .set("features", FEATURES)
 .set("arch", list(mt.ARCH))
 .set("scaler", "fixed_range")
 .integer("n_params", int(model.count_params()))
 .integer("n_train_rows", int(report.get("n_out", len(train_rows))))
 .integer("n_train_bp", int(train_rows["bp"].nunique()))
 .integer("n_epochs", len(history["loss"]) if history else 0)
 .num("norm_mean", NORM_MEAN, 4)
 .num("norm_std", NORM_STD, 4)
 .num("rmse_first_half", RMSE_H1, 3)
 .num("rmse_second_half", RMSE_H2, 3)
 .stats(stats, _keys, into="datasets")
 .save(res_store, "model", model_cfg, final=IS_FINAL))

In [ ]:
from src.utils import latex_export as lx

mx = lx.MacroExport("automatisch erzeugt aus mill_model.ipynb - nicht manuell editieren")
mx.comment("Parameter des statischen MLP")
mx.integer("millParamNeurons", mt.ARCH[0])
mx.integer("millParamNLayers", len(mt.ARCH))
mx.integer("millParamEpochs", mt.EPOCHS)
mx.integer("millParamBatch", mt.BATCH)
mx.integer("millParamValSplitPct", round(mt.VAL_SPLIT * 100))
mx.integer("millNParams", int(model.count_params()))
mx.integer("millNTrainRows", int(report.get("n_out", len(train_rows))))
mx.integer("millNTrainBp", int(train_rows["bp"].nunique()))

mx.comment("Modellfehler je Datensatz")
for _name, _tok in (("DOE1", "DoeOne"), ("DOE2", "DoeTwo"),
                    ("Produktion", "Production")):
    mx.num(f"millRmse{_tok}", float(stats.loc[_name, "rmse"]), 2)
    mx.num(f"millBias{_tok}", float(stats.loc[_name, "bias"]), 2)
    mx.num(f"millRsq{_tok}", float(stats.loc[_name, "r2"]), 2)

mx.comment("Fehlerentwicklung ueber den Produktionszeitraum")
mx.num("millRmseFirstHalf", RMSE_H1, 2)
mx.num("millRmseSecondHalf", RMSE_H2, 2)
mx.save(res_store, "model", model_cfg, final=IS_FINAL)

In [ ]:
print(f"model_cfg.id = {model_cfg.id}   -> RUN_ID in mill_context, "
      "mill_detection, mill_adaptation")